
# **Training: Transfer Learning-Based Kitchen Cleanliness Classification**
This notebook implements a Transfer Learning pipeline for binary image classification focused on kitchen cleanliness detection.

The objective is to classify induction stove images into two categories:

- `clean`
- `dirty`

The workflow includes:
- dataset loading
- transfer learning using MobileNetV3
- model training and fine-tuning
- model evaluation
- metric extraction and JSON export

The datasets are already preprocessed and divided into:

- training set
- validation set
- test set

with the following directory structure:

```text
dataset/
    train/
        clean/
        dirty/

    val/
        clean/
        dirty/

    test/
        clean/
        dirty/
````

The model uses a pretrained MobileNetV3 backbone initialized with ImageNet weights. Transfer learning is applied to leverage pretrained visual features while adapting the network to the kitchen cleanliness classification task.


# **Libraries used**

In [1]:
from pathlib import Path
import tensorflow as tf
import numpy as np
import pandas as pd
import cv2
import os
import tensorflow as tf
import json
import time

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


import matplotlib.pyplot as plt
import seaborn as sns

# **Auxiliar functions**

## **1) Load Dataset**

In [2]:
def load_dataset_for_cnn(
    dataset_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
):
    """
    Returns:
        train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names
    where:
        - train_ds, val_ds, test_ds: tf.data.Dataset objects for training, validation, and testing.
        - y_train, y_val, y_test: Numpy arrays containing the labels for the training, validation, and testing datasets.
        - class_names: List of class names corresponding to the labels.
    """

    dataset_dir = Path(dataset_dir)

    train_dir = dataset_dir / "train"
    val_dir = dataset_dir / "val"
    test_dir = dataset_dir / "test"

    
    # LOAD DATASETS
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=shuffle
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=False
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=False
    )

    # SAVE CLASS NAMES
    class_names = train_ds.class_names

    # NORMALIZATION
    normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

    train_ds = train_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    val_ds = val_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    test_ds = test_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    # EXTRACT LABELS
    y_train = np.concatenate([
        y.numpy() for _, y in train_ds
    ])

    y_val = np.concatenate([
        y.numpy() for _, y in val_ds
    ])

    y_test = np.concatenate([
        y.numpy() for _, y in test_ds
    ])

    # Flatten labels
    y_train = y_train.flatten()
    y_val = y_val.flatten()
    y_test = y_test.flatten()

  
    # PERFORMANCE
    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

    return (
        train_ds,
        val_ds,
        test_ds,
        y_train,
        y_val,
        y_test,
        class_names
    )

## **2) CNN pipeline** 

The implemented model is a sequential Convolutional Neural Network (CNN) designed for binary image classification (`clean` vs `dirty`).


# Architecture Table

| Layer Type | Parameters | Output Purpose |
|---|---|---|
| Input Layer | `(224, 224, 3)` | Receives RGB images resized to 224×224 |
| Conv2D | 32 filters, 3×3 kernel, ReLU | Extracts low-level visual features |
| MaxPooling2D | 2×2 pool size | Reduces spatial dimensions |
| Conv2D | 64 filters, 3×3 kernel, ReLU | Learns intermediate visual patterns |
| MaxPooling2D | 2×2 pool size | Downsampling |
| Conv2D | 128 filters, 3×3 kernel, ReLU | Learns higher-level semantic features |
| MaxPooling2D | 2×2 pool size | Further dimensionality reduction |
| Flatten | — | Converts feature maps into a 1D vector |
| Dense | 128 neurons, ReLU | Fully connected feature learning |
| Dropout | 0.5 | Reduces overfitting |
| Output Dense | 1 neuron, Sigmoid | Produces binary classification probability |



In [3]:
def build_and_train_mobilenetv3(
    train_ds,
    val_ds,
    input_shape=(224, 224, 3),
    num_classes=1,
    epochs=10,
    learning_rate=0.001
):
    """
    Builds and trains a Transfer Learning model using MobileNetV3
    as a frozen feature extractor for binary classification.

    Returns:
        model
        history
    """

    # ------------------------------------------------
    # PRETRAINED BACKBONE
    # ------------------------------------------------

    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )

    # Freeze feature extractor layers
    base_model.trainable = True

    for layer in base_model.layers[:-20]:
        layer.trainable = False

    # ------------------------------------------------
    # MODEL
    # ------------------------------------------------

    inputs = tf.keras.Input(shape=input_shape)

    # MobileNetV3 preprocessing
    x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)

    # Feature extraction
    x = base_model(x, training=False)

    # Global pooling instead of Flatten
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # ------------------------------------------------
    # MLP CLASSIFIER
    # ------------------------------------------------

    x = tf.keras.layers.Dense(
        128,
        activation="relu"
    )(x)

    x = tf.keras.layers.Dropout(0.5)(x)

    x = tf.keras.layers.Dense(
        64,
        activation="relu"
    )(x)

    x = tf.keras.layers.Dropout(0.3)(x)

    # ------------------------------------------------
    # OUTPUT LAYER
    # ------------------------------------------------

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="sigmoid"
    )(x)

    model = tf.keras.Model(inputs, outputs)

    # ------------------------------------------------
    # COMPILE
    # ------------------------------------------------

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

    model.summary()

    # ------------------------------------------------
    # TRAIN
    # ------------------------------------------------

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs
    )

    return model, history

## **3) Metrics**

### **1) Folders**
This function check if the paths already exist or create them.

In [4]:
def create_experiment_folders():

    models_dir = Path("models")
    results_dir = Path("results")

    models_dir.mkdir(exist_ok=True)
    results_dir.mkdir(exist_ok=True)

    return models_dir, results_dir


### **2) Save models**
This function check if the paths already exist or create them.

In [5]:
def save_model(model, model_name="cnn_model"):
    models_dir, _ = create_experiment_folders()
    model_path = models_dir / f"{model_name}.keras"
    model.save(model_path)
    print(f"\nModel saved at: {model_path}")
    return model_path


def save_metrics_json(
    metrics,
    save_path="results/metrics.json"
):
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved at: {save_path}")

This function print the confusion matrix in a more readable way

In [6]:
def plot_confusion_matrix_paper_style(
    y_test,
    y_pred,
    class_names,
    save_path="results/confusion_matrix.png"
):
    """
    Generates paper-style confusion matrix.
    """

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(
        cmap="Blues",
        ax=ax,
        colorbar=False
    )

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()
    print(f"Confusion matrix saved at: {save_path}")

In [7]:
def evaluate_model(
    model,
    test_ds,
    y_test,
    class_names,
    model_name="cnn_model",
    threshold=0.5
):
    """
    Evaluates CNN model.
    Saves:
        - model
        - metrics json
        - confusion matrix image
    """

    # CREATE FOLDERS
    models_dir, results_dir = create_experiment_folders()


    # INFERENCE
    start_time = time.time()
    y_probs = model.predict(test_ds)
    end_time = time.time()
    inference_time = end_time - start_time

    # PREDICTIONS
    y_probs = y_probs.flatten()
    y_pred = (y_probs >= threshold).astype(int)

    # METRICS
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_probs)
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )
    cm = confusion_matrix(
        y_test,
        y_pred
    )

    # RESULTS DICTIONARY
    metrics = {

        "model_name": model_name,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "auc": float(auc),
        "inference_time_seconds": float(
            inference_time
        ),
        "num_test_samples": int(
            len(y_test)
        ),
        "classification_report": report,
        "confusion_matrix": cm.tolist()
    }


    # PRINT RESULTS
    print("\n========== MODEL METRICS ==========\n")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")
    print(
        f"\nInference Time: "
        f"{inference_time:.4f} seconds"
    )
    print("\n========== CLASSIFICATION REPORT ==========\n")
    print(
        classification_report(
            y_test,
            y_pred
        )
    )

    # SAVE MODEL
    save_model(
        model=model,
        model_name=model_name
    )

    # SAVE METRICS JSON
    metrics_path = (
        results_dir /
        f"{model_name}_metrics.json"
    )

    save_metrics_json(
        metrics=metrics,
        save_path=metrics_path
    )

    # SAVE CONFUSION MATRIX
    cm_path = (
        results_dir /
        f"{model_name}_confusion_matrix.png"
    )

    plot_confusion_matrix_paper_style(
        y_test=y_test,
        y_pred=y_pred,
        class_names=class_names,
        save_path=cm_path
    )

    return metrics

In [8]:
ORIGINAL_SPLIT_PATH = "No_augmented_images_split"
AUGMENTED_SPLIT_PATH = "Augmented_images_split"

# **CNN: Original Data**

In [9]:
train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names = load_dataset_for_cnn(
    dataset_dir=ORIGINAL_SPLIT_PATH,
    image_size=(224, 224),
    batch_size=32
)

Found 209 files belonging to 2 classes.
Found 45 files belonging to 2 classes.
Found 46 files belonging to 2 classes.


In [10]:
model, history = build_and_train_mobilenetv3(
    train_ds=train_ds,
    val_ds=val_ds,
    input_shape=(224, 224, 3),
    epochs=15,
    learning_rate=1e-5
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,021,297 (3.90 MB)

 Trainable params: 432,913 (1.65 MB)

 Non-trainable params: 588,384 (2.24 MB)

Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 245ms/step - accuracy: 0.4928 - loss: 0.7479 - precision: 0.6374 - recall: 0.4427 - val_accuracy: 0.3778 - val_loss: 0.7564 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.4641 - loss: 0.7655 - precision: 0.5979 - recall: 0.4427 - val_accuracy: 0.3778 - val_loss: 0.7577 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 0.5407 - loss: 0.7106 - precision: 0.6522 - recall: 0.5725 - val_accuracy: 0.3778 - val_loss: 0.7587 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.5359 - loss: 0.7055 - precision: 0.6349 - recall: 0.6107 - val_accuracy: 0.3778 - val_loss: 0.7593 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step - accuracy: 0.4737 - loss: 0.7381 - precision: 0.5827 - recall: 0.5649 - val_accuracy: 

In [11]:
metrics = evaluate_model(
    model=model,
    test_ds=test_ds,
    y_test=y_test,
    class_names=class_names,
    model_name="mobilenet3_no_augmented"
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 865ms/step

========== MODEL METRICS ==========

Accuracy  : 0.3696
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000
AUC       : 0.4645

Inference Time: 1.8658 seconds

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         0.0       0.37      1.00      0.54        17
         1.0       0.00      0.00      0.00        29

    accuracy                           0.37        46
   macro avg       0.18      0.50      0.27        46
weighted avg       0.14      0.37      0.20        46



c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av


Model saved at: models\mobilenet3_no_augmented.keras
Metrics saved at: results\mobilenet3_no_augmented_metrics.json
Confusion matrix saved at: results\mobilenet3_no_augmented_confusion_matrix.png


# **CNN: Augmented Data**

In [12]:
train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names = load_dataset_for_cnn(
    dataset_dir=AUGMENTED_SPLIT_PATH,
    image_size=(224, 224),
    batch_size=32
)

Found 1254 files belonging to 2 classes.
Found 45 files belonging to 2 classes.
Found 46 files belonging to 2 classes.


In [13]:
model, history = build_and_train_mobilenetv3(
    train_ds=train_ds,
    val_ds=val_ds,
    input_shape=(224, 224, 3),
    epochs=15,
    learning_rate=1e-5
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,021,297 (3.90 MB)

 Trainable params: 432,913 (1.65 MB)

 Non-trainable params: 588,384 (2.24 MB)

Epoch 1/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 12s 166ms/step - accuracy: 0.6116 - loss: 0.7008 - precision_1: 0.6241 - recall_1: 0.9567 - val_accuracy: 0.6222 - val_loss: 0.6744 - val_precision_1: 0.6222 - val_recall_1: 1.0000
Epoch 2/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 96ms/step - accuracy: 0.6077 - loss: 0.6848 - precision_1: 0.6248 - recall_1: 0.9364 - val_accuracy: 0.6222 - val_loss: 0.6755 - val_precision_1: 0.6222 - val_recall_1: 1.0000
Epoch 3/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 114ms/step - accuracy: 0.6069 - loss: 0.6707 - precision_1: 0.6266 - recall_1: 0.9224 - val_accuracy: 0.6222 - val_loss: 0.6762 - val_precision_1: 0.6222 - val_recall_1: 1.0000
Epoch 4/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 34s 794ms/step - accuracy: 0.6220 - loss: 0.6644 - precision_1: 0.6383 - recall_1: 0.9160 - val_accuracy: 0.6222 - val_loss: 0.6769 - val_precision_1: 0.6222 - val_recall_1: 1.0000
Epoch 5/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 39s 710ms/step - accuracy: 0.6300 - loss: 0.6532 - precision_1: 0.6381 - recall_1: 0.9466

In [14]:
metrics = evaluate_model(
    model=model,
    test_ds=test_ds,
    y_test=y_test,
    class_names=class_names,
    model_name="mobilenet3_augmented"
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 861ms/step

========== MODEL METRICS ==========

Accuracy  : 0.6304
Precision : 0.6304
Recall    : 1.0000
F1 Score  : 0.7733
AUC       : 0.5233

Inference Time: 1.9950 seconds

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        17
         1.0       0.63      1.00      0.77        29

    accuracy                           0.63        46
   macro avg       0.32      0.50      0.39        46
weighted avg       0.40      0.63      0.49        46



c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war


Model saved at: models\mobilenet3_augmented.keras
Metrics saved at: results\mobilenet3_augmented_metrics.json
Confusion matrix saved at: results\mobilenet3_augmented_confusion_matrix.png
